In [ ]:
!pip install Faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.6 MB/s eta 0:00:00


In [ ]:
from faker import Faker
import random
import pandas as pd

fake = Faker('pt_BR')

data = []

for i in range(1000):
    data.append({
        "id_cliente": random.randint(1, 100),
        "valor_transacao": round(random.uniform(10, 20000), 2),
        "data": fake.date_this_month(),
        "cidade": fake.city()
    })

df = pd.DataFrame(data)
df.to_csv("dados_fraude.csv", index=False)

1. INICIALIZAÇÃO (PySpark)

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Fraude") \
    .getOrCreate()

2. LEITURA DOS DADOS

In [ ]:
from pyspark.sql.functions import col, to_date

df = spark.read.csv(
    "dados_fraude.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

df = df.withColumn(
    "data",
    to_date(col("data"), "dd/MM/yyyy")  # 🔥 formato correto agora
)

df.show()

+----------+---------------+----------+--------------------+
|id_cliente|valor_transacao|      data|              cidade|
+----------+---------------+----------+--------------------+
|        85|        2802.21|2026-04-28|             da Mota|
|        10|        9844.93|2026-04-28|      Lopes do Campo|
|        26|       14469.07|2026-04-05|               Silva|
|        19|        2134.48|2026-04-07|              Vieira|
|        10|         4027.0|2026-04-16| Mendes de Fernandes|
|        75|       13927.57|2026-04-16|  Aparecida de Minas|
|        13|       12431.66|2026-04-02|            Monteiro|
|        51|        5054.42|2026-04-11|              Pastor|
|        66|       13214.36|2026-04-10|    Peixoto da Praia|
|        92|        7168.79|2026-04-19|     Castro da Serra|
|        18|        1233.59|2026-04-01|Rezende dos Dourados|
|        68|       12806.49|2026-04-25|               Ramos|
|        19|       17478.29|2026-04-12|             Andrade|
|        34|       13817

3. TRATAMENTO DE DADOS ETL

In [ ]:
## Tratamento dados
df.printSchema()

root
 |-- id_cliente: integer (nullable = true)
 |-- valor_transacao: double (nullable = true)
 |-- data: date (nullable = true)
 |-- cidade: string (nullable = true)



In [ ]:
df = df.fillna({
    "valor_transacao": 0
})

4. ANÁLISE EXPLORATÓRIA

In [ ]:
### Analise exploratoria
## Total por cliente
from pyspark.sql.functions import sum

df_total = df.groupBy("id_cliente") \
    .agg(sum("valor_transacao").alias("total_gasto"))

df_total.show()

+----------+------------------+
|id_cliente|       total_gasto|
+----------+------------------+
|        31| 92786.68000000001|
|        85|          41210.99|
|        65|136819.33000000002|
|        53|          79529.18|
|        78|108350.45000000001|
|        34|165370.81999999998|
|        81|163330.66000000003|
|        28|         134138.81|
|        76|           87817.2|
|        26|103151.32999999999|
|        27| 65164.07000000001|
|        44|          90782.34|
|        12|          60522.69|
|        91|          90013.12|
|        22|         108638.78|
|        93|          85971.23|
|        47|          76490.16|
|         1|          67335.35|
|        52|         123349.08|
|        13|158261.81000000003|
+----------+------------------+
only showing top 20 rows


In [ ]:
## Media por cidade
from pyspark.sql.functions import avg

df_cidade = df.groupBy("cidade") \
    .agg(avg("valor_transacao").alias("media_valor"))

df_cidade.show()

+--------------------+------------------+
|              cidade|       media_valor|
+--------------------+------------------+
|Casa Grande de Minas|          10136.57|
|  Araújo de Nogueira|          10498.28|
|  Gonçalves do Norte|           5048.53|
|   Mendonça da Serra|          18708.79|
|             Camargo|           7634.64|
|        Pires Alegre|          17920.92|
| Cardoso de Teixeira|           2828.02|
|             Correia|11476.053333333335|
| Ferreira de Sampaio|          10925.53|
|    da Mota de Souza|          19962.98|
|     Araújo da Prata|           4255.65|
|    Cardoso de Goiás|           8946.05|
| Farias dos Dourados|          12010.73|
|            Monteiro|14343.503333333334|
|              Castro|11860.806666666665|
|          Nascimento|           7937.07|
|     Mendonça do Sul|            7817.4|
|Pimenta de Albuqu...|          16387.71|
|      Moreira do Sul|          13490.44|
|Rezende dos Dourados|           1233.59|
+--------------------+------------

DETECÇÃO DE FRAUDE (INCLUSÃO REGRAS DE NEGÓCIO)

In [ ]:
## Detecção de Fraude
df = df.withColumn(
    "flag_valor_alto",
    (col("valor_transacao") > 10000).cast("int")
)

In [ ]:
## Frequencia por cliente
from pyspark.sql.functions import count

df_freq = df.groupBy("id_cliente") \
    .agg(count("*").alias("qtd_transacoes"))

df = df.join(df_freq, on="id_cliente", how="left")

df = df.withColumn(
    "flag_frequencia",
    (col("qtd_transacoes") > 10).cast("int")
)

In [ ]:
## Score de risco
df = df.withColumn(
    "score_risco",
    col("flag_valor_alto") + col("flag_frequencia")
)

DATAFRAME FINAL

In [ ]:
## Resultado final
df.select(
    "id_cliente",
    "valor_transacao",
    "cidade",
    "score_risco"
).show()

+----------+---------------+--------------------+-----------+
|id_cliente|valor_transacao|              cidade|score_risco|
+----------+---------------+--------------------+-----------+
|        85|        2802.21|             da Mota|          0|
|        10|        9844.93|      Lopes do Campo|          1|
|        26|       14469.07|               Silva|          1|
|        19|        2134.48|              Vieira|          1|
|        10|         4027.0| Mendes de Fernandes|          1|
|        75|       13927.57|  Aparecida de Minas|          2|
|        13|       12431.66|            Monteiro|          2|
|        51|        5054.42|              Pastor|          0|
|        66|       13214.36|    Peixoto da Praia|          2|
|        92|        7168.79|     Castro da Serra|          1|
|        18|        1233.59|Rezende dos Dourados|          1|
|        68|       12806.49|               Ramos|          1|
|        19|       17478.29|             Andrade|          2|
|       

In [ ]:
## Ordem por risco
df.orderBy(col("score_risco").desc()).show()

+----------+---------------+----------+------------------+---------------+--------------+---------------+-----------+
|id_cliente|valor_transacao|      data|            cidade|flag_valor_alto|qtd_transacoes|flag_frequencia|score_risco|
+----------+---------------+----------+------------------+---------------+--------------+---------------+-----------+
|         8|       14073.91|2026-04-10| Silveira de Porto|              1|            14|              1|          2|
|        97|       14480.39|2026-04-19|              Lima|              1|            11|              1|          2|
|        50|       11510.07|2026-04-03|          Caldeira|              1|            15|              1|          2|
|        13|       12431.66|2026-04-02|          Monteiro|              1|            17|              1|          2|
|        34|       15008.47|2026-04-28|   Sampaio de Leão|              1|            15|              1|          2|
|        19|       17478.29|2026-04-12|           Andrad

In [ ]:
## Filtrar suspeitos
df.filter(col("score_risco") > 0).show()

+----------+---------------+----------+--------------------+---------------+--------------+---------------+-----------+
|id_cliente|valor_transacao|      data|              cidade|flag_valor_alto|qtd_transacoes|flag_frequencia|score_risco|
+----------+---------------+----------+--------------------+---------------+--------------+---------------+-----------+
|        10|        9844.93|2026-04-28|      Lopes do Campo|              0|            16|              1|          1|
|        26|       14469.07|2026-04-05|               Silva|              1|             8|              0|          1|
|        19|        2134.48|2026-04-07|              Vieira|              0|            14|              1|          1|
|        10|         4027.0|2026-04-16| Mendes de Fernandes|              0|            16|              1|          1|
|        75|       13927.57|2026-04-16|  Aparecida de Minas|              1|            11|              1|          2|
|        13|       12431.66|2026-04-02| 

Aplicação ML

In [ ]:
df = df.withColumn(
    "fraude",
    (col("score_risco") > 0).cast("int")
)

In [ ]:
## Converter cidade
from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(inputCol="cidade", outputCol="cidade_index")
df = indexer.fit(df).transform(df)

In [ ]:
## Selecionar Features
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["valor_transacao", "qtd_transacoes", "cidade_index"],
    outputCol="features"
)

df = assembler.transform(df)

In [ ]:
## Modelo regressão logistica
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="fraude"
)

modelo = lr.fit(df)

In [ ]:
## Previsão
predicoes = modelo.transform(df)

predicoes.select(
    "id_cliente",
    "features",
    "fraude",
    "prediction",
    "probability"
).show()

+----------+--------------------+------+----------+--------------------+
|id_cliente|            features|fraude|prediction|         probability|
+----------+--------------------+------+----------+--------------------+
|        85|  [2802.21,5.0,56.0]|     0|       0.0|[0.99894800531957...|
|        10|[9844.93,16.0,107.0]|     1|       1.0|[3.83690388106185...|
|        26| [14469.07,8.0,24.0]|     1|       1.0|[0.04254445771403...|
|        19| [2134.48,14.0,71.0]|     1|       1.0|[0.02912267444859...|
|        10| [4027.0,16.0,352.0]|     1|       1.0|[7.56141174989084...|
|        75|[13927.57,11.0,15...|     1|       1.0|[0.00149751106847...|
|        13|[12431.66,17.0,84.0]|     1|       1.0|[2.86505165063012...|
|        51| [5054.42,10.0,11.0]|     0|       1.0|[0.42618491636117...|
|        66|[13214.36,14.0,41...|     1|       1.0|[4.92032139301175...|
|        92|[7168.79,15.0,221.0]|     1|       1.0|[4.96652654859646...|
|        18|[1233.59,12.0,452.0]|     1|       1.0|

In [ ]:
## Avaliação
from pyspark.ml.evaluation import BinaryClassificationEvaluator

avaliador = BinaryClassificationEvaluator(
    labelCol="fraude"
)

auc = avaliador.evaluate(predicoes)

print(f"AUC: {auc}")

AUC: 0.9692538969522345
